# Gold-2: property questions

- **Q3** Properties with repeated service problems
- **Q4** Properties with unusually high complaint activity vs similar buildings
- **Q5** Property types with a wide variety of city-service needs

## 1. Shared setup

In [ ]:
import findspark
findspark.init()

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
DEFAULT_OUTPUT_DIR = "/home/ubuntu/spark-project-data/dev"
DEFAULT_SILVER_PATH = f"{DEFAULT_OUTPUT_DIR}/silver/nyc311_enriched"
DEFAULT_GOLD_OUTPUT = f"{DEFAULT_OUTPUT_DIR}/gold"
GOLD_BASE_COLUMNS = [
    # request
    "request_id", "created_at", "closed_at", "request_year", "request_month",
    "status", "resolution_hours", "lifecycle_quality_status",
    # what / who
    "complaint_type", "agency",
    # where (Member A)
    "borough", "community_board", "incident_zip",
    # property (Member B)
    "bbl", "pluto_match_status", "pluto_land_use", "pluto_building_class",
    "pluto_residential_units", "pluto_yearbuilt",
]


def create_spark_session(app_name="DAT535-NYC311-PLUTO-Gold", cores_max="12"):
    """Spark session for the cluster.

    Tip: when both members run notebooks at the same time, use cores_max="6"
    each. With "12" the first notebook takes every core and the second waits.
    """
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("spark://192.168.18.168:7077")
        .config("spark.driver.host", "192.168.18.168")
        .config("spark.cores.max", cores_max)
        .config("spark.executor.cores", "4")
        .config("spark.executor.memory", "6g")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "32")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    return spark


def load_gold_base(spark, silver_path=DEFAULT_SILVER_PATH, sample_pct=None):
    """LEVEL-1 CACHE: read Silver once, select Gold columns, persist.

    sample_pct: e.g. 1 = deterministic 1% of requests (same rows every run),
                useful in notebooks. None = full data (use in gold.py).
    Returns (base_df, row_count). The count() fills the cache and doubles as
    the "Silver is not empty" check.
    """
    base = (
        spark.read.parquet(silver_path)
        .select(*GOLD_BASE_COLUMNS)
        .withColumn("request_date", F.to_date("created_at"))
    )

    if sample_pct is not None:
        base = base.filter(F.abs(F.hash("request_id")) % 100 < sample_pct)

    base = base.persist(StorageLevel.MEMORY_AND_DISK)
    row_count = base.count()
    if row_count == 0:
        raise ValueError(f"Silver is empty: {silver_path}")
    print(f"[Level-1 cache] Gold base cached: {row_count} rows", flush=True)
    return base, row_count


def write_table(df, path, partition_by=None):
    """Write one Gold table as Parquet (overwrite)."""
    writer = df.write.mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.parquet(path)
    print(f"  wrote {path}", flush=True)

In [ ]:
spark = create_spark_session(app_name="DAT535-Gold-MemberB", cores_max="6")

SAMPLE_PCT = 1      # 1% while developing; None = full data 

## 2. Load Silver – Level-1 cache

In [ ]:
# LEVEL-1 CACHE 
base, n = load_gold_base(spark, DEFAULT_SILVER_PATH, sample_pct=SAMPLE_PCT)
base.printSchema()

## 3. Rules (thresholds)

In [ ]:
REFERENCE_YEAR = 2026
REPEAT_MIN_REQUESTS = 3
REPEAT_MIN_MONTHS = 2
PEER_MIN_PROPERTIES = 10
HIGH_ACTIVITY_RATIO = 3.0
VARIETY_MIN_PROPERTIES = 30

## 4. Property profile – Level-2 cache
Expensive `groupBy` on all matched requests, reused by Q3, Q4 and Q5 → cache it.

In [ ]:
def unit_band(units_col):
    """Size band from residential units (NULL/0 kept as their own band)."""
    return (
        F.when(units_col.isNull(), "unknown")
        .when(units_col == 0, "0 (non-res)")
        .when(units_col <= 2, "1-2")
        .when(units_col <= 10, "3-10")
        .when(units_col <= 50, "11-50")
        .otherwise("51+")
    )


def get_matched_requests(base):
    """Requests whose valid BBL matched a PLUTO tax lot.

    No extra cache: this is a cheap filter on the already-cached base.
    """
    return base.filter(F.col("pluto_match_status") == "matched")


def build_property_profile(matched):
    """One row per matched property with request totals and PLUTO attributes."""
    return (
        matched.groupBy("bbl").agg(
            F.count("*").alias("total_requests"),
            F.countDistinct("complaint_type").alias("unique_complaint_types"),
            F.countDistinct("agency").alias("unique_agencies"),
            F.countDistinct("request_year", "request_month").alias("active_months"),
            F.min("created_at").alias("first_request"),
            F.max("created_at").alias("last_request"),
            F.round(F.avg("resolution_hours"), 2).alias("avg_resolution_hours"),
            F.first("borough", ignorenulls=True).alias("borough"),
            F.first("pluto_land_use", ignorenulls=True).alias("land_use"),
            F.first("pluto_building_class", ignorenulls=True).alias("building_class"),
            F.first("pluto_residential_units", ignorenulls=True).alias("residential_units"),
            F.first("pluto_yearbuilt", ignorenulls=True).alias("year_built"),
        )
        .withColumn("unit_band", unit_band(F.col("residential_units")))
        .withColumn(
            "requests_per_res_unit",   # safe division
            F.when(F.col("residential_units") > 0,
                   F.round(F.col("total_requests") / F.col("residential_units"), 2)),
        )
        .withColumn(
            "building_age",            # year 0 / NULL = unknown
            F.when(F.col("year_built") > 0, F.lit(REFERENCE_YEAR) - F.col("year_built")),
        )
    )

In [ ]:
matched = get_matched_requests(base)        # cheap filter, no cache needed

profile = build_property_profile(matched).persist(StorageLevel.MEMORY_AND_DISK)
print("properties:", profile.count())       # count() fills the cache
profile.orderBy(F.desc("total_requests")).show(10, truncate=False)

## 5. Q3 – Repeated service problems
Repeated = same complaint type at the same property ≥ `REPEAT_MIN_REQUESTS` times, in ≥ `REPEAT_MIN_MONTHS` different months.

In [ ]:
def build_repeat_problems(matched):
    """One row per (property, complaint type) that keeps coming back."""
    return (
        matched.filter(F.col("complaint_type").isNotNull())
        .groupBy("bbl", "complaint_type").agg(
            F.count("*").alias("request_count"),
            F.countDistinct("request_year", "request_month").alias("active_months"),
            F.min("created_at").alias("first_request"),
            F.max("created_at").alias("last_request"),
        )
        .withColumn("span_days", F.datediff("last_request", "first_request"))
        .filter(
            (F.col("request_count") >= REPEAT_MIN_REQUESTS)
            & (F.col("active_months") >= REPEAT_MIN_MONTHS)
        )
    )


def build_repeat_summary(repeat_problems, profile):
    """One row per property with repeated problems, plus its top repeated type."""
    # Lab 3 window function: rank complaint types inside each property
    w = Window.partitionBy("bbl").orderBy(F.desc("request_count"), "complaint_type")
    top_type = (
        repeat_problems.withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .select("bbl",
                F.col("complaint_type").alias("top_repeated_type"),
                F.col("request_count").alias("top_repeated_count"))
    )
    summary = repeat_problems.groupBy("bbl").agg(
        F.count("*").alias("repeated_problem_types"),
        F.sum("request_count").alias("repeated_requests"),
        F.max("span_days").alias("longest_span_days"),
    )
    return (
        summary.join(top_type, "bbl")
        .join(profile.select("bbl", "total_requests", "borough", "land_use",
                             "building_class", "residential_units"), "bbl", "left")
        .withColumn("repeat_share",
                    F.round(F.col("repeated_requests") / F.col("total_requests") * 100, 2))
        .orderBy(F.desc("repeated_requests"))
    )

In [ ]:
repeat_problems = build_repeat_problems(matched).cache()   # used twice below
print("repeated (property, complaint) pairs:", repeat_problems.count())
repeat_problems.orderBy(F.desc("request_count")).show(10, truncate=False)

repeat_summary = build_repeat_summary(repeat_problems, profile)
repeat_summary.show(10, truncate=False)

## 6. Q4 – Unusually high activity vs similar buildings
Peer group = borough + land use + residential-unit size band. Flag = above peer p90 **and** ≥ `HIGH_ACTIVITY_RATIO` × peer median, only in groups with ≥ `PEER_MIN_PROPERTIES` properties.

In [ ]:
PEER_COLS = ["peer_borough", "peer_land_use", "unit_band"]


def build_peer_comparison(profile):
    """Compare each property with its peer group (borough + land use + size band)."""
    with_peer = (
        profile
        .withColumn("peer_borough", F.coalesce("borough", F.lit("UNKNOWN")))
        .withColumn("peer_land_use", F.coalesce("land_use", F.lit("UNKNOWN")))
    )

    # Peer statistics (small table: one row per peer group)
    peer_stats = with_peer.groupBy(*PEER_COLS).agg(
        F.count("*").alias("peer_properties"),
        F.percentile_approx("total_requests", 0.5).alias("peer_median_requests"),
        F.percentile_approx("total_requests", 0.9).alias("peer_p90_requests"),
        F.round(F.avg("total_requests"), 2).alias("peer_avg_requests"),
    )

    # Lab 3: broadcast the small lookup table in the join
    # Lab 3: window function percent_rank inside the peer group
    w = Window.partitionBy(*PEER_COLS).orderBy("total_requests")
    return (
        with_peer.join(F.broadcast(peer_stats), PEER_COLS, "left")
        .withColumn("peer_percentile", F.round(F.percent_rank().over(w) * 100, 2))
        .withColumn(
            "activity_ratio",          # safe division
            F.when(F.col("peer_median_requests") > 0,
                   F.round(F.col("total_requests") / F.col("peer_median_requests"), 2)),
        )
        .withColumn(
            "is_unusually_high",
            (F.col("peer_properties") >= PEER_MIN_PROPERTIES)
            & (F.col("total_requests") > F.col("peer_p90_requests"))
            & (F.col("activity_ratio") >= HIGH_ACTIVITY_RATIO),
        )
        .select("bbl", *PEER_COLS, "building_class", "residential_units",
                "total_requests", "peer_properties", "peer_median_requests",
                "peer_p90_requests", "peer_avg_requests", "peer_percentile",
                "activity_ratio", "is_unusually_high")
    )

In [ ]:
peer = build_peer_comparison(profile)
peer.filter("is_unusually_high").orderBy(F.desc("activity_ratio")).show(10, truncate=False)
peer.groupBy("is_unusually_high").count().show()

## 7. Q5 – Property types with a wide variety of service needs

In [ ]:
def build_property_type_variety(matched, profile):
    """One row per property type (land use + building class)."""
    # Average variety per property (from the cached profile)
    per_property = (
        profile
        .groupBy("land_use", "building_class").agg(
            F.count("*").alias("properties"),
            F.sum("total_requests").alias("total_requests"),
            F.round(F.avg("unique_complaint_types"), 2).alias("avg_types_per_property"),
            F.round(F.avg("unique_agencies"), 2).alias("avg_agencies_per_property"),
        )
    )
    # Total variety across the whole property type (needs request-level rows)
    whole_type = (
        matched.groupBy(
            F.col("pluto_land_use").alias("land_use"),
            F.col("pluto_building_class").alias("building_class"),
        ).agg(
            F.countDistinct("complaint_type").alias("distinct_complaint_types"),
            F.countDistinct("agency").alias("distinct_agencies"),
        )
    )
    return (
        per_property.join(whole_type, ["land_use", "building_class"], "left")
        .withColumn("enough_properties", F.col("properties") >= VARIETY_MIN_PROPERTIES)
        .orderBy(F.desc("enough_properties"), F.desc("avg_types_per_property"))
    )

In [ ]:
variety = build_property_type_variety(matched, profile)
variety.filter("enough_properties").show(15, truncate=False)

## 8. Save to DEFAULT_GOLD_OUTPUT

In [ ]:
write_table(profile, f"{DEFAULT_GOLD_OUTPUT}/property_profile")
write_table(repeat_summary, f"{DEFAULT_GOLD_OUTPUT}/property_repeat_summary")
write_table(peer, f"{DEFAULT_GOLD_OUTPUT}/property_peer_comparison")
write_table(variety, f"{DEFAULT_GOLD_OUTPUT}/property_type_variety")

## 9. Clean up – unpersist, then stop

In [ ]:
repeat_problems.unpersist()
profile.unpersist()
base.unpersist()
spark.stop()